<a href="https://colab.research.google.com/github/mnsbharadwaj/AI-NLP/blob/master/Retrieval_Augmented_Geneartion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# -*- coding: utf-8 -*-
"""RAG_Implementation_Complete.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1abc123
"""

# %% [markdown]
# # Retrieval-Augmented Generation (RAG) - Complete Implementation
#
# This notebook provides a complete implementation of RAG using free, open-source models.

# %% [markdown]
# ## 1. Installation and Setup

# %%
!pip install -q langchain faiss-cpu sentence-transformers chromadb pypdf pdfplumber
!pip install -q transformers torch accelerate
!pip install -U langchain-community

# %%
# Import all required libraries
import os
import requests
import numpy as np
from typing import List, Dict, Any

# LangChain components
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.llms import HuggingFacePipeline

# Transformers for local LLM
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM

# PDF processing
import pdfplumber

print("✅ All libraries imported successfully")

# %% [markdown]
# ## 2. Document Processing Pipeline

# %%
def setup_document_processor():
    """Download and process a sample PDF document"""

    # Download a sample research paper about RAG
    def download_pdf():
        url = "https://arxiv.org/pdf/2305.15334.pdf"
        response = requests.get(url)
        with open("rag_paper.pdf", "wb") as f:
            f.write(response.content)
        print("📄 PDF downloaded successfully")
        return "rag_paper.pdf"

    # Load and chunk the PDF
    def load_and_chunk_pdf(pdf_path: str, chunk_size: int = 500, chunk_overlap: int = 50):
        print("📖 Loading PDF document...")
        loader = PyPDFLoader(pdf_path)
        documents = loader.load()
        print(f"📑 Loaded {len(documents)} pages")

        # Create text splitter
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            length_function=len,
            separators=["\n\n", "\n", ". ", " ", ""]
        )

        # Split documents into chunks
        chunks = text_splitter.split_documents(documents)
        print(f"✂️  Split into {len(chunks)} chunks (size: {chunk_size}, overlap: {chunk_overlap})")

        return chunks

    pdf_path = download_pdf()
    chunks = load_and_chunk_pdf(pdf_path)

    return pdf_path, chunks

# Initialize document processing
pdf_path, document_chunks = setup_document_processor()

# %% [markdown]
# ## 3. Vector Database Setup

# %%
def create_vector_database(chunks):
    """Create embeddings and vector store for the documents"""

    print("🔮 Initializing embedding model...")

    # Use sentence transformers for embeddings
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        model_kwargs={'device': 'cpu'},  # Use CPU for compatibility
        encode_kwargs={'normalize_embeddings': True}
    )

    print("🏗️ Creating vector database...")

    # Create FAISS vector store
    vector_store = FAISS.from_documents(
        documents=chunks,
        embedding=embeddings
    )

    print("✅ Vector database created successfully")
    print(f"📊 Vector store contains {vector_store.index.ntotal} vectors")

    return vector_store, embeddings

# Create vector database
vector_store, embeddings = create_vector_database(document_chunks)

# %% [markdown]
# ## 4. Local LLM Setup

# %%
def setup_local_llm():
    """Setup a local LLM using HuggingFace models"""

    print("🤖 Loading local language model...")

    try:
        # Use a small, efficient model for demonstration
        model_name = "microsoft/DialoGPT-medium"

        # Load tokenizer and model
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForCausalLM.from_pretrained(model_name)

        # Add padding token if missing
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        # Create text generation pipeline
        text_generation_pipeline = pipeline(
            "text-generation",
            model=model,
            tokenizer=tokenizer,
            max_new_tokens=150,
            temperature=0.7,
            do_sample=True,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )

        # Wrap in LangChain pipeline
        llm = HuggingFacePipeline(pipeline=text_generation_pipeline)
        print("✅ Local LLM loaded successfully")
        return llm

    except Exception as e:
        print(f"❌ Error loading model: {e}")
        print("🔄 Using fallback mock LLM...")
        return setup_mock_llm()

def setup_mock_llm():
    """Fallback mock LLM for demonstration"""
    from langchain.llms.fake import FakeListLLM

    # Pre-defined responses for common RAG questions
    responses = [
        "Retrieval-Augmented Generation (RAG) combines information retrieval with language generation to produce more accurate responses.",
        "RAG works by first retrieving relevant documents from a knowledge base, then using a language model to generate answers based on that context.",
        "The main benefits of RAG include reduced hallucinations, access to up-to-date information, and better factuality in generated responses.",
        "RAG differs from fine-tuning by allowing dynamic knowledge updates without retraining the entire model.",
        "Vector databases store document embeddings for efficient similarity search in RAG systems."
    ]

    return FakeListLLM(responses=responses)

# Initialize LLM
llm = setup_local_llm()

# %% [markdown]
# ## 5. RAG Chain Implementation

# %%
def create_rag_system(vector_store, llm):
    """Create the complete RAG system"""

    print("⚙️ Configuring RAG system...")

    # Create a custom prompt template
    prompt_template = """
You are an AI assistant that answers questions based on the provided context.

Please use the following context to answer the question. If the context doesn't contain enough information to answer the question fully, please indicate what information is missing.

Context:
{context}

Question: {question}

Please provide a comprehensive and accurate answer based on the context above.
Answer: """

    # Create prompt template
    PROMPT = PromptTemplate(
        template=prompt_template,
        input_variables=["context", "question"]
    )

    # Create retriever
    retriever = vector_store.as_retriever(
        search_type="similarity",
        search_kwargs={"k": 3}  # Retrieve top 3 most relevant chunks
    )

    # Create RAG chain
    rag_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",  # Simple "stuff all context" approach
        retriever=retriever,
        return_source_documents=True,
        chain_type_kwargs={
            "prompt": PROMPT,
            "verbose": True
        }
    )

    print("✅ RAG system configured successfully")
    return rag_chain

# Create RAG system
rag_chain = create_rag_system(vector_store, llm)

# %% [markdown]
# ## 6. Testing the RAG System

# %%
def test_rag_system(rag_chain, test_questions):
    """Test the RAG system with sample questions"""

    print("🧪 Testing RAG System")
    print("=" * 60)

    results = []

    for i, question in enumerate(test_questions, 1):
        print(f"\n🔍 Test {i}: {question}")
        print("-" * 40)

        try:
            # Execute RAG query
            result = rag_chain({"query": question})

            # Display results
            print(f"💡 Answer: {result['result']}")
            print(f"📚 Sources used: {len(result['source_documents'])}")

            # Show source preview
            for j, doc in enumerate(result['source_documents'][:2]):  # Show first 2 sources
                print(f"   Source {j+1}: {doc.page_content[:100]}...")

            results.append(result)

        except Exception as e:
            print(f"❌ Error: {e}")
            # Fallback: show similar documents
            similar_docs = vector_store.similarity_search(question, k=2)
            print(f"🔄 Fallback - Similar documents:")
            for j, doc in enumerate(similar_docs):
                print(f"   Doc {j+1}: {doc.page_content[:100]}...")

    return results

# Test questions
test_questions = [
    "What is Retrieval-Augmented Generation?",
    "How does RAG work?",
    "What are the benefits of using RAG?",
    "How is RAG different from fine-tuning?"
]

# Run tests
test_results = test_rag_system(rag_chain, test_questions)

# %% [markdown]
# ## 7. Document Content Analysis

# %%
def analyze_document_content(pdf_path):
    """Analyze and display PDF content"""

    print("📊 Document Content Analysis")
    print("=" * 50)

    with pdfplumber.open(pdf_path) as pdf:
        # Basic information
        print(f"Total pages: {len(pdf.pages)}")

        # Analyze first few pages
        for page_num in range(min(3, len(pdf.pages))):
            page = pdf.pages[page_num]
            text = page.extract_text() or ""

            print(f"\n📄 Page {page_num + 1}:")
            print("-" * 30)
            if text:
                # Show first 300 characters
                preview = text[:300] + "..." if len(text) > 300 else text
                print(f"Content: {preview}")
                print(f"Stats: {len(text)} chars, {len(text.split())} words")
            else:
                print("No extractable text found")

        # Overall statistics
        total_text = ""
        for page in pdf.pages:
            text = page.extract_text() or ""
            total_text += text + " "

        print(f"\n📈 Document Statistics:")
        print(f"Total characters: {len(total_text):,}")
        print(f"Total words: {len(total_text.split()):,}")
        print(f"Average words per page: {len(total_text.split()) / len(pdf.pages):.0f}")

# Analyze document
analyze_document_content(pdf_path)

# %% [markdown]
# ## 8. Vector Search Demonstration

# %%
def demonstrate_vector_search(vector_store):
    """Show how vector search works"""

    print("🎯 Vector Search Demonstration")
    print("=" * 50)

    search_queries = [
        "machine learning",
        "language models",
        "information retrieval",
        "neural networks"
    ]

    for query in search_queries:
        print(f"\n🔍 Query: '{query}'")

        # Perform similarity search
        results = vector_store.similarity_search(query, k=2)

        for i, doc in enumerate(results):
            print(f"   Result {i+1}: {doc.page_content[:150]}...")
            print(f"   Metadata: Page {doc.metadata.get('page', 'N/A')}")
            print()

# Demonstrate vector search
demonstrate_vector_search(vector_store)

# %% [markdown]
# ## 9. Interactive RAG Query Interface

# %%
def interactive_rag_interface(rag_chain, vector_store):
    """Interactive interface for testing RAG"""

    print("🎮 Interactive RAG Interface")
    print("=" * 50)
    print("Type your questions about the document (type 'quit' to exit)")
    print()

    while True:
        try:
            question = input("🤔 Your question: ").strip()

            if question.lower() in ['quit', 'exit', 'q']:
                print("👋 Goodbye!")
                break

            if not question:
                continue

            print("⏳ Processing...")

            # Get RAG response
            result = rag_chain({"query": question})

            print(f"\n💡 Answer: {result['result']}")
            print(f"📚 Sources referenced: {len(result['source_documents'])}")

            # Show source information
            for i, doc in enumerate(result['source_documents']):
                print(f"\n   Source {i+1} (Page {doc.metadata.get('page', 'N/A')}):")
                print(f"   {doc.page_content[:200]}...")

            print("\n" + "="*50)

        except KeyboardInterrupt:
            print("\n👋 Goodbye!")
            break
        except Exception as e:
            print(f"❌ Error: {e}")
            print("🔄 Trying similarity search fallback...")

            # Fallback to simple search
            docs = vector_store.similarity_search(question, k=2)
            if docs:
                print("📖 Relevant document chunks:")
                for i, doc in enumerate(docs):
                    print(f"   {i+1}. {doc.page_content[:200]}...")
            else:
                print("   No relevant documents found.")

# Uncomment to run interactive interface
# interactive_rag_interface(rag_chain, vector_store)

# %% [markdown]
# ## 10. RAG System Evaluation

# %%
def evaluate_rag_performance(rag_chain, vector_store):
    """Evaluate RAG system performance"""

    print("📊 RAG System Evaluation")
    print("=" * 50)

    evaluation_questions = [
        {
            "question": "What is the main topic of this document?",
            "expected_keywords": ["RAG", "Retrieval-Augmented", "Generation"]
        },
        {
            "question": "How does retrieval work in RAG?",
            "expected_keywords": ["vector", "similarity", "search", "embedding"]
        },
        {
            "question": "What are the advantages mentioned?",
            "expected_keywords": ["hallucination", "accuracy", "knowledge", "update"]
        }
    ]

    for eval_item in evaluation_questions:
        question = eval_item["question"]
        expected_keywords = eval_item["expected_keywords"]

        print(f"\n🔍 Question: {question}")
        print(f"   Expected keywords: {expected_keywords}")

        try:
            # Get RAG response
            result = rag_chain({"query": question})
            answer = result["result"].lower()

            # Check for expected keywords
            found_keywords = []
            for keyword in expected_keywords:
                if keyword.lower() in answer:
                    found_keywords.append(keyword)

            # Calculate score
            score = len(found_keywords) / len(expected_keywords)

            print(f"   ✅ Found keywords: {found_keywords}")
            print(f"   📊 Score: {score:.1%} ({len(found_keywords)}/{len(expected_keywords)})")
            print(f"   💡 Answer preview: {answer[:100]}...")

        except Exception as e:
            print(f"   ❌ Error: {e}")

# Run evaluation
evaluate_rag_performance(rag_chain, vector_store)

# %% [markdown]
# ## 11. Complete RAG Pipeline Summary

# %%
def summarize_rag_pipeline():
    """Provide a comprehensive summary of the RAG pipeline"""

    print("🎯 RAG Pipeline Summary")
    print("=" * 60)

    pipeline_steps = [
        {
            "step": 1,
            "name": "Document Ingestion",
            "description": "Load and process PDF documents",
            "components": "PyPDFLoader, TextSplitter",
            "output": f"{len(document_chunks)} text chunks"
        },
        {
            "step": 2,
            "name": "Embedding Generation",
            "description": "Convert text to vector representations",
            "components": "HuggingFace Embeddings",
            "output": "384-dimensional vectors"
        },
        {
            "step": 3,
            "name": "Vector Storage",
            "description": "Store vectors for efficient search",
            "components": "FAISS Vector Database",
            "output": f"{vector_store.index.ntotal} vectors stored"
        },
        {
            "step": 4,
            "name": "Query Processing",
            "description": "Convert user questions to vectors",
            "components": "Same embedding model",
            "output": "Query vector for similarity search"
        },
        {
            "step": 5,
            "name": "Retrieval",
            "description": "Find most relevant document chunks",
            "components": "Similarity search",
            "output": "Top-K most relevant chunks"
        },
        {
            "step": 6,
            "name": "Generation",
            "description": "LLM generates answer using context",
            "components": "Local LLM + Prompt engineering",
            "output": "Coherent, context-based answer"
        }
    ]

    for step in pipeline_steps:
        print(f"\n{step['step']}. {step['name']}")
        print(f"   Description: {step['description']}")
        print(f"   Components: {step['components']}")
        print(f"   Output: {step['output']}")

    print(f"\n🎉 RAG Pipeline is ready with {len(document_chunks)} document chunks!")
    print("💡 You can ask questions about the document using the interactive interface.")

# Display summary
summarize_rag_pipeline()

# %% [markdown]
# ## Key Concepts Explained

# %%
# %% [markdown]
# ### Why We Send Context to LLM (Detailed Explanation)

# %%
# Demonstration of why context is sent to LLM
def demonstrate_context_importance():
    print("🤔 WHY CONTEXT IS SENT TO LLM")
    print("=" * 60)

    # Example query
    query = "What are the main components of a RAG system?"

    print(f"Question: {query}")
    print("\n1. 🔍 VECTOR DB SEARCH (Finds relevant information):")

    # What vector DB returns
    similar_docs = vector_store.similarity_search(query, k=2)
    for i, doc in enumerate(similar_docs):
        print(f"   Chunk {i+1}: {doc.page_content[:100]}...")

    print("\n2. 🧠 LLM PROCESSING (Synthesizes and explains):")
    print("   - Reads and understands all retrieved chunks")
    print("   - Identifies key information about RAG components")
    print("   - Synthesizes coherent answer from multiple sources")
    print("   - Structures response in human-readable format")

    print("\n3. 💡 FINAL ANSWER (Intelligent synthesis):")
    print("   'A RAG system typically consists of three main components:")
    print("   1. Retriever: Finds relevant documents using vector similarity'")
    print("   2. Generator: LLM that produces answers based on context'")
    print("   3. Vector Database: Stores and searches document embeddings'")

    print("\n🎯 KEY INSIGHT: Vector DB finds pieces, LLM assembles the puzzle!")

demonstrate_context_importance()

# %% [markdown]
# ## Conclusion

# %%
# %% [markdown]
# ### 🎉 RAG Implementation Complete!
#
# **What we've built:**
# - ✅ Document processing pipeline
# - ✅ Vector database with semantic search
# - ✅ Local LLM integration
# - ✅ Complete RAG system
# - ✅ Testing and evaluation framework
#
# **Key takeaways:**
# 1. **RAG enhances LLMs** with external knowledge
# 2. **Vector databases enable** efficient similarity search
# 3. **Context + LLM = Intelligent answers**
# 4. **No API keys needed** with open-source models
#
# **Next steps:**
# - Try the interactive interface with your own questions
# - Replace the sample PDF with your own documents
# - Experiment with different chunk sizes and models

print("🚀 RAG implementation completed successfully!")
print("💡 Use the interactive_rag_interface() function to ask questions about the document!")

✅ All libraries imported successfully
📄 PDF downloaded successfully
📖 Loading PDF document...
📑 Loaded 18 pages
✂️  Split into 141 chunks (size: 500, overlap: 50)
🔮 Initializing embedding model...


/tmp/ipython-input-2465726681.py:102: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Device set to use cpu
/tmp/ipython-input-2465726681.py:158: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=text_generation_pipeline)


✅ Local LLM loaded successfully
⚙️ Configuring RAG system...
✅ RAG system configured successfully
🧪 Testing RAG System

🔍 Test 1: What is Retrieval-Augmented Generation?
----------------------------------------


> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:

You are an AI assistant that answers questions based on the provided context.

Please use the following context to answer the question. If the context doesn't contain enough information to answer the question fully, please indicate what information is missing.

Context:
to query the LLMs. Similarly, GPT-Index refers to the retrieval model text-davinci-003 from
OpenAI. Like BM25, each API call is indexed as an individual document, and the most relevant
document, given a user query, is retrieved and appended to the user prompt. Lastly, we include
an Oracle retriever, which serves two purposes: first, to identify the potential for performance
improvement through more efficient

/tmp/ipython-input-2465726681.py:256: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = rag_chain({"query": question})



> Finished chain.

> Finished chain.
💡 Answer: 
You are an AI assistant that answers questions based on the provided context.

Please use the following context to answer the question. If the context doesn't contain enough information to answer the question fully, please indicate what information is missing.

Context:
to query the LLMs. Similarly, GPT-Index refers to the retrieval model text-davinci-003 from
OpenAI. Like BM25, each API call is indexed as an individual document, and the most relevant
document, given a user query, is retrieved and appended to the user prompt. Lastly, we include
an Oracle retriever, which serves two purposes: first, to identify the potential for performance
improvement through more efficient retrievers, and second, to assist users who know which API

to training, can be used for inference in two modes: zero-shot and with retrieval. In zero-shot, this
prompt (with NO further prompt tuning) is fed to the Gorilla LLM model when then returns the
API call that